# 2. Indice di qualità dell'aria
Calcola un indice di qualità dell'aria complessivo per ciascuna provincia, normalizzando i valori degli inquinanti con `MinMaxScaler()` e calcolando l'indice come somma dei valori normalizzati.

In [ ]:
import geopandas as gpd
import h3pandas
from shapely.geometry import Point, Polygon
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
import leafmap

# Importazione e pulizia dati

In [ ]:
#importazione dati
dati = pd.read_csv("/Users/giulia/Documents/geopandas/Air-quality-Lombardy-2025/data/Dati_sensori_aria_2025.csv")
stazioni = gpd.read_file("/Users/giulia/Documents/geopandas/Air-quality-Lombardy-2025/data/Stazioni qualità dell aria_20260826/geo_export_1e1f2c5a-263d-4a72-908d-7477842f54c0.shp")
province = gpd.read_file("/Users/giulia/Documents/geopandas/Air-quality-Lombardy-2025/data/Limiti amministrativi Province 2020 con aggiornamenti DbT_PGT_20260826/geo_export_97ffb683-dd2e-40cd-b28e-c03e599be95f.shp")

In [ ]:
#pulizia dati di qualità dell'aria, delle stazioni e delle province
import functions as f
dati_puliti = f.pulizia_dati(dati)
stazioni_pulite = f.pulizia_stazioni(stazioni)
province = province.to_crs("EPSG:7791")
province = province.rename(columns={'sigla': 'provincia'})

# Media giornaliera per sensore

In [ ]:
#calcolo della media giornaliera per sensore
dati_giornalieri = dati_puliti.groupby(["idsensore", dati_puliti["Data"].dt.date])["Valore"].mean().reset_index()
dati_giornalieri.columns = ["idsensore", "Data", "Valore_medio"]

# Merge stazioni e dati 

In [ ]:
#merge stazioni e dati 
dati_stazioni_giornalieri = pd.merge(
    dati_giornalieri, stazioni_pulite, on='idsensore', how='left'
)

In [ ]:
#Conversione in un GeoDataFrame
dati_stazioni_giornalieri = gpd.GeoDataFrame(dati_stazioni_giornalieri, geometry='geometry', crs=stazioni_pulite.crs)

In [ ]:
# Visualizzazione del GeoDataFrame
dati_stazioni_giornalieri.explore()

# Media per provincia

In [ ]:
dati_provincia_giornalieri = dati_stazioni_giornalieri.groupby(["provincia", "inquinante"])["Valore_medio"].mean().reset_index()

# Pivot table

In [ ]:
df_pivot = dati_provincia_giornalieri.pivot(index="provincia", columns="inquinante", values="Valore_medio").reset_index()
df_pivot

In [ ]:
#seleziono i principali inquinanti da analizzare (NOX, NO2, SO2, CO, O3, PM10, PM2.5 e benzene)
columns = ['provincia', 'Ossidi di Azoto', 'Biossido di Azoto', 'Biossido di Zolfo', 'Monossido di Carbonio', 'Ozono', 'PM10 (SM2005)', 'Particelle sospese PM2.5', 'Benzene']
df_pivot = df_pivot[columns]

# Merge dati e province

In [ ]:
#merge dati e province 
df = pd.merge(
    df_pivot, province, on='provincia', how='left'
)

In [ ]:
#conversione in un GeoDataFrame
df = gpd.GeoDataFrame(df, geometry='geometry', crs=province.crs)

# Normalizzazione e indice di qualità dell'aria

In [ ]:
#normalizzazione dei valori degli inquinanti con `MinMaxScaler()`
scaler = MinMaxScaler()
inquinanti = ['Ossidi di Azoto', 'Biossido di Azoto', 'Biossido di Zolfo', 'Monossido di Carbonio', 'Ozono', 'PM10 (SM2005)', 'Particelle sospese PM2.5', 'Benzene']
df[inquinanti] = scaler.fit_transform(df[inquinanti])
df = df.fillna(0)

In [ ]:
#calcolo dell'indice di qualità dell'aria come somma dei valori normalizzati degli inquinanti
df['score'] = df['Benzene']+df['Biossido di Azoto']+df['Biossido di Zolfo']+df['Monossido di Carbonio']+df['Ossidi di Azoto']+df['Ozono']+df['PM10 (SM2005)']+df['Particelle sospese PM2.5']
df = df.replace(0, np.nan)

In [ ]:
df

# Visualizzazione del valore dell'indice di qualità dell'aria per le province Lombarde

Le province con indice di qualità dell'aria peggiore sono MI, MB e BS; qelle con indice migliore sono SO e LC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

df.plot(
    column="score",
    cmap="Reds",
    legend=True,
    edgecolor="black",
    linewidth=0.5,
    ax=ax,
)

ax.set_title("Indice di qualità dell'aria per provincia — Lombardia")
ax.set_axis_off()

plt.tight_layout()
plt.show()

NameError: name 'm' is not defined